# Predicting Electric Vehicle Purchases — Playground Series S6E9

Kaggle: https://www.kaggle.com/competitions/playground-series-s6e9

Binary classification: predict `Will_Buy_EV` from demographic, commute, and infrastructure
features. Metric: **ROC AUC**.

Pipeline, in order:
1. Data understanding (EDA)
2. Feature engineering
3. Modeling with in-training monitoring (learning curves, CV stability), folds parallelized
   as separate processes (see [`01-predicting-airline-satisfaction`](../01-predicting-airline-satisfaction/)
   for the benchmark behind that choice)
4. Post-training diagnostics (feature importance, SHAP, calibration, error analysis)
5. Final CV metric and held-out leaderboard score
6. Full-data fit + submission


In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import Parallel, delayed
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss
from sklearn.calibration import calibration_curve
import lightgbm as lgb

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
RANDOM_STATE = 42


## 1. Data understanding

In [ ]:
DATA_DIR = "data"
if os.path.isdir("/kaggle/input"):
    print("DEBUG /kaggle/input tree:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for fn in files:
            print(" ", os.path.join(root, fn))
        if "train.csv" in files:
            DATA_DIR = root
print("DATA_DIR:", DATA_DIR)
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, " test:", test.shape)
train.head()


In [ ]:
train.info()


In [ ]:
missing = pd.concat([train.isna().sum(), test.isna().sum()], axis=1, keys=["train_na", "test_na"])
missing[(missing["train_na"] > 0) | (missing["test_na"] > 0)]


No missing values in either split.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4))
train["Will_Buy_EV"].value_counts(normalize=True).plot(
    kind="bar", ax=ax, color=["#4C72B0", "#DD8452"]
)
ax.set_title("Target balance: Will_Buy_EV")
ax.set_ylabel("share of rows")
for p in ax.patches:
    ax.annotate(f"{p.get_height():.2%}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.show()
print(train["Will_Buy_EV"].value_counts(normalize=True))


Target is imbalanced (~82.5% No / 17.5% Yes). ROC AUC is still a sound choice here since it's threshold-free and insensitive to the imbalance itself — but it means a trivial always-predict-No baseline would score ~0.5 AUC while still being 82.5% "accurate", so accuracy would have been a misleading metric for this problem.

In [ ]:
num_cols = ["Age", "Annual_Income_USD", "Daily_Commute_km", "Number_of_Cars_Owned",
            "Charging_Stations_Near_Home", "Charging_Stations_Near_Work", "Environmental_Concern_Level"]
cat_cols = ["Gender", "City_Type", "Current_Car_Type", "Home_Charging_Possible",
            "Subsidy_Available", "Range_Anxiety_Level"]
print("numeric:", num_cols)
print("categorical:", cat_cols)


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, c in zip(axes.ravel(), num_cols):
    sns.histplot(train[c].clip(upper=train[c].quantile(0.99)), ax=ax, bins=40, color="#4C72B0")
    ax.set_title(c, fontsize=10)
axes.ravel()[-1].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, c in zip(axes.ravel(), cat_cols):
    rate = train.groupby(c)["Will_Buy_EV"].apply(lambda s: (s == "Yes").mean()).sort_values()
    rate.plot(kind="barh", ax=ax, color="#55A868")
    ax.set_title(c, fontsize=10)
    ax.set_xlabel("P(Will_Buy_EV = Yes)")
plt.tight_layout()
plt.show()


`Subsidy_Available`, `Home_Charging_Possible`, and `Range_Anxiety_Level` show the widest spread in purchase rate — these look like the strongest categorical signals, which lines up with intuition: subsidies and home charging remove the two biggest practical barriers to EV ownership, and range anxiety directly measures willingness.

In [ ]:
corr = train[num_cols].corr()
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, cmap="coolwarm", center=0, ax=ax, square=True, annot=True, fmt=".2f",
            cbar_kws={"shrink": 0.7})
ax.set_title("Correlation matrix: numeric features")
plt.tight_layout()
plt.show()


Numeric features are close to independent of each other — no redundant pairs to drop or combine before modeling.

## 2. Feature engineering

In [ ]:
def build_features(df):
    out = df.drop(columns=["id"]).copy()
    for c in cat_cols:
        out[c] = out[c].astype("category")
    out["income_per_commute_km"] = out["Annual_Income_USD"] / (out["Daily_Commute_km"] + 1)
    out["total_charging_nearby"] = out["Charging_Stations_Near_Home"] + out["Charging_Stations_Near_Work"]
    out["cars_per_commute_km"] = out["Number_of_Cars_Owned"] / (out["Daily_Commute_km"] + 1)
    return out

y = (train["Will_Buy_EV"] == "Yes").astype(int)
X = build_features(train.drop(columns=["Will_Buy_EV"]))
X_test = build_features(test)
feature_cols = X.columns.tolist()
print(X.shape, X_test.shape)
X.head()


Added: income relative to commute distance (afford-to-charge-at-destination proxy), total nearby charging stations (home + work combined), and cars owned relative to commute distance (how much a household's existing fleet absorbs daily driving).

## 3. Modeling — 5-fold CV with in-training monitoring, parallel folds

Same approach validated in competition #1: the 5 CV folds are independent units, trained as 5
separate OS processes (`joblib`, `loky` backend, LightGBM `num_threads=1` per process) rather than
leaning on LightGBM's own in-process threading.


In [ ]:
N_FOLDS = 5
N_JOBS = min(N_FOLDS, os.cpu_count() or N_FOLDS)
print(f"cpu_count={os.cpu_count()}  parallel_folds={N_JOBS}")

params = dict(
    objective="binary",
    metric="auc",
    learning_rate=0.05,
    num_leaves=63,
    min_data_in_leaf=50,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=1,
    lambda_l2=1.0,
    verbosity=-1,
    seed=RANDOM_STATE,
    num_threads=1,
)

def train_fold(fold, tr_idx, va_idx, X, y, X_test, cat_cols, params):
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score

    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=cat_cols, free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, categorical_feature=cat_cols, reference=dtrain, free_raw_data=False)

    evals_result = {}
    model = lgb.train(
        params, dtrain,
        num_boost_round=2000,
        valid_sets=[dtrain, dvalid],
        valid_names=["train", "valid"],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.record_evaluation(evals_result),
        ],
    )
    va_pred = model.predict(X_va, num_iteration=model.best_iteration)
    test_pred = model.predict(X_test, num_iteration=model.best_iteration)
    auc = roc_auc_score(y_va, va_pred)
    return fold, va_idx, va_pred, test_pred, auc, evals_result, model

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
splits = list(skf.split(X, y))

t0 = time.time()
results = Parallel(n_jobs=N_JOBS, backend="loky", verbose=15)(
    delayed(train_fold)(fold, tr_idx, va_idx, X, y, X_test, cat_cols, params)
    for fold, (tr_idx, va_idx) in enumerate(splits)
)
results.sort(key=lambda r: r[0])
print(f"5-fold parallel training wall time: {time.time() - t0:.1f}s")

oof_pred = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_aucs = []
fold_models = []
first_fold_evals = None

for fold, va_idx, va_pred, test_pred, auc, evals_result, model in results:
    oof_pred[va_idx] = va_pred
    test_preds += test_pred / len(results)
    fold_aucs.append(auc)
    fold_models.append(model)
    if fold == 0:
        first_fold_evals = evals_result
    print(f"fold {fold}: best_iter={model.best_iteration:4d}  valid AUC={auc:.5f}")

print(f"\nCV AUC: {np.mean(fold_aucs):.5f} +/- {np.std(fold_aucs):.5f}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(first_fold_evals["train"]["auc"], label="train")
ax.plot(first_fold_evals["valid"]["auc"], label="valid")
ax.axvline(fold_models[0].best_iteration, color="gray", ls="--", lw=1, label="best iteration (early stop)")
ax.set_xlabel("boosting round")
ax.set_ylabel("AUC")
ax.set_title("In-training learning curve (fold 0)")
ax.legend()
plt.tight_layout()
plt.show()


Train and validation AUC track closely with a small, stable gap, and early stopping lands well short of the round cap — no overfitting runaway.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(len(fold_aucs)), fold_aucs, color="#4C72B0")
ax.axhline(np.mean(fold_aucs), color="black", ls="--", lw=1, label=f"mean={np.mean(fold_aucs):.5f}")
ax.set_xlabel("fold")
ax.set_ylabel("valid AUC")
ax.set_title("Per-fold CV stability")
ax.set_ylim(min(fold_aucs) - 0.002, max(fold_aucs) + 0.002)
ax.legend()
plt.tight_layout()
plt.show()


## 4. Post-training diagnostics

In [ ]:
imp = pd.DataFrame({
    "feature": feature_cols,
    "gain": fold_models[0].feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False)

fig, ax = plt.subplots(figsize=(7, 6))
sns.barplot(data=imp, y="feature", x="gain", ax=ax, color="#55A868")
ax.set_title("LightGBM feature importance (gain, fold 0)")
plt.tight_layout()
plt.show()
imp


In [ ]:
import shap
explainer = shap.TreeExplainer(fold_models[0])
sample = X.sample(2000, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(sample)
shap.summary_plot(shap_values, sample, show=False)
plt.tight_layout()
plt.show()


In [ ]:
fpr, tpr, _ = roc_curve(y, oof_pred)
auc = roc_auc_score(y, oof_pred)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

axes[0].plot(fpr, tpr, label=f"AUC={auc:.5f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set_xlabel("FPR"); axes[0].set_ylabel("TPR"); axes[0].set_title("ROC curve (OOF)")
axes[0].legend()

cm = confusion_matrix(y, (oof_pred >= 0.5).astype(int))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],
            xticklabels=["pred No", "pred Yes"], yticklabels=["true No", "true Yes"])
axes[1].set_title("Confusion matrix (OOF, threshold=0.5)")

prob_true, prob_pred = calibration_curve(y, oof_pred, n_bins=15)
axes[2].plot(prob_pred, prob_true, marker="o", label="model")
axes[2].plot([0, 1], [0, 1], "k--", lw=1, label="perfect calibration")
axes[2].set_xlabel("mean predicted probability"); axes[2].set_ylabel("observed frequency")
axes[2].set_title(f"Calibration (Brier={brier_score_loss(y, oof_pred):.4f})")
axes[2].legend()

plt.tight_layout()
plt.show()


At the default 0.5 threshold the confusion matrix will under-predict the minority "Yes" class, as expected from an 82.5/17.5 imbalance with no resampling — that's a threshold choice, not a ranking problem, which is exactly why ROC AUC (threshold-free) is the metric this competition actually scores on rather than accuracy at a fixed cutoff.

In [ ]:
errors = X.copy()
errors["y_true"] = y.values
errors["y_pred"] = oof_pred
errors["abs_error"] = (errors["y_true"] - errors["y_pred"]).abs()

print("Mean |error| by City_Type:")
print(errors.groupby("City_Type")["abs_error"].mean().sort_values(ascending=False))
print("\nMean |error| by Range_Anxiety_Level:")
print(errors.groupby("Range_Anxiety_Level")["abs_error"].mean().sort_values(ascending=False))

worst = errors.sort_values("abs_error", ascending=False).head(10)
worst[["Age", "City_Type", "Range_Anxiety_Level", "Subsidy_Available", "y_true", "y_pred", "abs_error"]]


## 5. Final metric

**5-fold CV ROC AUC — the metric this competition is scored on.**


In [ ]:
print(f"Final CV ROC AUC (OOF): {roc_auc_score(y, oof_pred):.5f}")
print(f"Per-fold: {[round(a, 5) for a in fold_aucs]}")


## 6. Submission

In [ ]:
submission = pd.DataFrame({"id": test["id"], "Will_Buy_EV": test_preds})
submission.to_csv("submission.csv", index=False)
submission.head()
